In [1]:
%cd ../..

/home/bhkuser/bhklab/katy/readii_2_roqc


In [2]:
import pandas as pd
from readii_2_roqc.utils.loaders import load_dataset_config, load_signature_config
from readii.io.writers.correlation_writer import CorrelationWriter
from readii_2_roqc.utils.settings import image_type_iterator
from readii.process.subset import getOnlyPyradiomicsFeatures
from damply import dirs
from readii.analyze.plot_correlation import saveCorrelationHeatmap
import re
from readii.utils import logger
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np
from pathlib import Path

In [3]:
def add_pyrad_feature_class_labels(corr_hm:plt.Figure) -> None:

    # Put on the feature class labels, remove the individual feature labels
    corr_ax = corr_hm.axes[0]

    corr_ax.set_xticklabels(labels=[])
    corr_ax.set_xticks([])
    corr_ax.set_yticklabels(labels=[])
    corr_ax.set_yticks([])

    feature_positions = [7, 25, 45, 65, 81, 97, 106]
    feature_types = ["shape", "firstorder", "glcm", "glrlm", "glszm", "gldm", "\nngtdm"]

    feature_label_x = corr_ax.secondary_xaxis(location=0)
    feature_label_x.set_xticks(feature_positions, labels=feature_types)
    feature_label_x.tick_params('x', length=0)

    # Lines between classes:
    division_lines_x = corr_ax.secondary_xaxis(location=0)
    division_lines_x.set_xticks([0, 16, 34, 58, 74, 90, 105, 109], labels=[])
    division_lines_x.tick_params('x', length=10, width = 1.5)

    feature_label_y = corr_ax.secondary_yaxis(location=0)
    feature_label_y.set_yticks(list(reversed(feature_positions)), labels=list(reversed(feature_types)))
    feature_label_y.tick_params('y', length=0)

    division_lines_y = corr_ax.secondary_yaxis(location=0)
    division_lines_y.set_yticks([0, 16, 34, 58, 74, 90, 105, 109], labels=[])
    division_lines_y.tick_params('y', length=10, width = 1.5)
    
    corr_ax.xaxis.labelpad = 30
    corr_ax.yaxis.labelpad = 50

    return corr_hm

In [4]:
def plotCorrelationHeatmap(correlation_matrix_df:pd.DataFrame,
                           diagonal:bool = False,
                           triangle:str = "lower",
                           cmap:str = "nipy_spectral",
                           xlabel:str = "",
                           ylabel:str = "",
                           title:str = "",
                           subtitle:str = "",
                           show_tick_labels:bool = False,
                           vmin:float = -1.0,
                           vmax:float = 1.0
                           ) -> plt.Figure:
    """Plot a correlation dataframe as a heatmap.

    Parameters
    ----------
    correlation_matrix_df : pd.DataFrame
        Dataframe containing the correlation matrix to plot.
    diagonal : bool, optional
        Whether to only plot half of the matrix. The default is False.
    triangle : str, optional
        Which triangle half of the matrix to plot. Either "lower" or "upper". 
        The default is "lower".
    xlabel : str, optional
        Label for the x-axis. The default is "".
    ylabel : str, optional
        Label for the y-axis. The default is "".
    title : str, optional
        Title for the plot. The default is "".
    subtitle : str, optional
        Subtitle for the plot. The default is "".
    show_tick_labels : bool, optional
        Whether to show the tick labels on the x and y axes. These would be the feature names. The default is False.

    Returns
    -------
    corr_fig : matplotlib.pyplot.figure
        Figure object containing a Seaborn heatmap.
    """
    if diagonal:
        logger.debug(f"Creating {triangle} traingle mask for diagonal correlation plot.")
        # Set up mask for hiding half the matrix in the plot
        if triangle == "lower":
            # Mask out the upper right triangle half of the matrix
            mask = np.triu(correlation_matrix_df, 1)
        elif triangle == "upper":
            # Mask out the lower left triangle half of the matrix
            mask = np.tril(correlation_matrix_df)
        else:
            msg = f"If diagonal is True, triangle must be either 'lower' or 'upper'. Got {triangle}."
            logger.exception(msg)
            raise ValueError()
    else:
        logger.debug("Creating full square correlation matrix plot.")
        # The entire correlation matrix will be visisble in the plot
        mask = None
    
    # Set a default title if one is not provided
    if not title:
        title = "Correlation Heatmap"

    # Set up figure and axes for the plot
    corr_fig, corr_ax = plt.subplots()

    # Plot the correlation matrix
    try:
        corr_ax = sns.heatmap(correlation_matrix_df,
                             mask = mask,
                             cmap=cmap,
                             vmin=vmin,
                             vmax=vmax,
                             xticklabels=True,
                             yticklabels=True)
    except Exception as e:
        msg = f"Error generating heatmap: {e}"
        logger.exception(msg)
        raise e
    
    if not show_tick_labels:
        # Remove the individual feature names from the axes
        corr_ax.set_xticklabels(labels=[])
        corr_ax.set_yticklabels(labels=[])
    else:
        corr_ax.set_xticklabels(corr_ax.get_xticklabels(), fontsize=4, ha='center')
        corr_ax.set_yticklabels(corr_ax.get_yticklabels(), fontsize=4, va='center')

        # sec2 = corr_ax.secondary_xaxis(location=0)
        # sec2.set_xticks()

    # Set axis labels
    corr_ax.set_xlabel(xlabel)
    corr_ax.set_ylabel(ylabel)
    
    # Set title and subtitle
    # Suptitle is the super title, which will be above the title
    plt.title(subtitle, fontsize=12)
    plt.suptitle(title, fontsize=14)
    
    return corr_fig

In [5]:
def get_pattern_subset_features(feature_dataframe: pd.DataFrame,
                                x_subset_keyword: str,
                                y_subset_keyword: str,
                                include_volume: bool = False,
                                include_shape: bool = False
                                ) -> pd.DataFrame:
    column_pattern = ".*" if x_subset_keyword is None else x_subset_keyword
    index_pattern = ".*" if y_subset_keyword is None else y_subset_keyword

    if include_shape:
        shape_pattern = "original_shape_"
        feature_subset = feature_dataframe.loc[feature_dataframe.index.str.contains(shape_pattern) | feature_dataframe.index.str.contains(index_pattern),
                                               feature_dataframe.columns.str.contains(shape_pattern) | feature_dataframe.columns.str.contains(column_pattern)]
    else:
        # select rows and columns of dataframe that have keywords in their names from dataframe
        feature_subset = feature_dataframe.loc[feature_dataframe.index.str.contains(index_pattern), 
                                            feature_dataframe.columns.str.contains(column_pattern)]

    if feature_subset.empty:
        message = f"Index and/or column pattern not found in the correlation matrix. Index pattern: {index_pattern}. Column pattern: {column_pattern}"
        raise ValueError(message)
        
    # remove subset_keywords prefix from index and columns
    x_pattern = re.compile(rf'.*{x_subset_keyword}_')
    feature_subset.columns = feature_subset.columns.str.replace(x_pattern, '', regex=True)

    y_pattern = re.compile(rf'.*{y_subset_keyword}_')
    feature_subset.index = feature_subset.index.str.replace(y_pattern, '', regex=True)

    if include_volume:
        # Get the mesh volume feature correlation row
        volume_correlations = feature_dataframe.loc['original_shape_MeshVolume', feature_subset.columns.to_list()]
        # Append the mesh volume correlation row to the bottom of the dataframe, matching the columns
        feature_subset = pd.concat([feature_subset, volume_correlations.to_frame().T])
        feature_subset = feature_subset.rename(index={'original_shape_MeshVolume': 'MeshVolume'})

    return feature_subset
    

In [6]:
def make_corr_figure(full_dataset_name:str,
                     correlations:pd.DataFrame,
                     correlation_method:str = "pearson", 
                     cmap:str = 'coolwarm',
                     x_subset_keyword: str = None,
                     y_subset_keyword: str = None,
                     include_volume: bool = False,
                     correlation_type: str = "self",
                     x_readii_itype:str = "original_full",
                     y_readii_itype:str | None = None,
                     signature: str|None = None,
                     overwrite: bool = False
                     ) -> tuple[plt.Figure, Path]:
    
    if x_subset_keyword is None and y_subset_keyword is None:
        corr_subset = correlations
        x_subset_keyword = "All"
        y_subset_keyword = "All"
    else:
        column_pattern = ".*" if x_subset_keyword is None else x_subset_keyword
        index_pattern = ".*" if y_subset_keyword is None else y_subset_keyword
        # select rows and columns of dataframe that have keywords in their names from dataframe
        corr_subset = correlations.loc[correlations.index.str.contains(index_pattern), 
                                       correlations.columns.str.contains(column_pattern)]

        if corr_subset.empty:
            message = f"Index and/or column pattern not found in the correlation matrix. Index pattern: {index_pattern}. Column pattern: {column_pattern}"
            raise ValueError(message)
            
        # remove subset_keywords prefix from index and columns
        x_pattern = re.compile(rf'.*{x_subset_keyword}_')
        corr_subset.columns = corr_subset.columns.str.replace(x_pattern, '', regex=True)

        y_pattern = re.compile(rf'.*{y_subset_keyword}_')
        corr_subset.index = corr_subset.index.str.replace(y_pattern, '', regex=True)

    if include_volume:
        # Get the mesh volume feature correlation row
        volume_correlations = correlations.loc['original_shape_MeshVolume', corr_subset.columns.to_list()]
        # Append the mesh volume correlation row to the bottom of the dataframe, matching the columns
        corr_subset = pd.concat([corr_subset, volume_correlations.to_frame().T])
        corr_subset = corr_subset.rename(index={'original_shape_MeshVolume': 'MeshVolume'})

    match correlation_type:
        case "self":
            title = f"Self-Correlation for {full_dataset_name}"
            subtitle = f"{' '.join(x_readii_itype.split('_'))} image"
            diagonal = True
            if signature:
                xlabel, ylabel = signature, signature
                feature_types = [f"{x_readii_itype}_{signature}"]
            else:
                xlabel, ylabel = x_subset_keyword, y_subset_keyword
                feature_types = [f"{x_readii_itype}_{x_subset_keyword}"]

            
        case "cross":
            title = f"Cross-Correlation for {full_dataset_name}"
            subtitle = f"{' '.join(y_readii_itype.split('_'))} image vs {' '.join(x_readii_itype.split('_'))} image"
            diagonal = False
            if signature:
                xlabel, ylabel = signature, signature
                feature_types = [f"{y_readii_itype}_{signature}_vs_{x_readii_itype}_{signature}"]
            else:
                xlabel, ylabel = x_subset_keyword, y_subset_keyword
                feature_types = [f"{y_readii_itype}_{y_subset_keyword}_vs_{x_readii_itype}_{x_subset_keyword}"]

        case _:
            raise ValueError(f"Unsupported correlation_type: {correlation_type}")

    # Get just the feature class for the corr heatmap
    # corr_subset.columns = corr_subset.columns.str.replace("_.*", "", regex=True)
    # corr_subset.index = corr_subset.index.str.replace("_.*", "", regex=True)
    # Plot correlations between features
    corr_hm = plotCorrelationHeatmap(corr_subset,
                         diagonal = diagonal,
                         cmap = cmap,
                         xlabel = f"{xlabel} Features",
                         ylabel = f"{ylabel} Features",
                         title = title,
                         subtitle = subtitle,
                         show_tick_labels=True
                         )
    # Resize heatmap figure
    corr_hm.set_size_inches(12,10)

    # remove the axis labels from the diagonal features (except if volume has been added)
    if diagonal:
        corr_hm.figure.axes[0].yaxis.get_major_ticks()[0].draw = lambda *args:None
        if not include_volume:
            corr_hm.figure.axes[0].xaxis.get_major_ticks()[-1].draw = lambda *args:None

    
    save_path = saveCorrelationHeatmap(corr_hm, 
                                       correlation_directory = dirs.RESULTS / full_dataset_name / "visualization" / "feature_correlation" / correlation_type / signature if signature else "",
                                       cmap = cmap,
                                       feature_types=feature_types,
                                       correlation_type=correlation_method,
                                       overwrite = overwrite)
    

    return corr_hm, save_path


In [7]:
def self_correlate(dataset: str,
                   correlation_method:str,
                   readii_itype:str = "original_full",
                   overwrite:bool = False,
                   plot:bool = False,
                   plot_cmap:str = 'coolwarm',
                   plot_subset_keyword:str = None,
                   plot_signature:str|None = None, # name of the signature to use for labelling
                   plot_include_volume:bool = False,
                   plot_overwrite:bool = False,
                   ) -> pd.DataFrame | tuple[pd.DataFrame, plt.Figure]:
    dataset_config, dataset_name, full_dataset_name = load_dataset_config(dataset)
    extract_method = dataset_config['EXTRACTION']['METHOD']
    extract_settings = dataset_config['EXTRACTION']['CONFIG']

    # Set up CorrelationWriter from readii
    corr_matrix_writer = CorrelationWriter(root_directory = dirs.RESULTS / full_dataset_name / "correlation" / "self" / extract_method / extract_settings,
                                           filename_format = "{Readii_iType}_{CorrelationMethod}.csv",
                                           overwrite = overwrite,
                                           create_dirs = True
    )

    corr_matrix_outpath = corr_matrix_writer.resolve_path(Readii_iType = readii_itype, CorrelationMethod=correlation_method)
    if corr_matrix_outpath.exists() and not overwrite:
        print(f"Correlation matrix already exists at {corr_matrix_outpath}. Skipping computation, loading existing matrix.")
        correlations = pd.read_csv(corr_matrix_outpath, index_col=0)
    
    else:
        print(f"Computing correlation matrix for {dataset_name} with method {correlation_method}...")

        features_path = dirs.RESULTS / full_dataset_name / "features" / extract_method / 'original_512_512_n' / extract_settings / f"{readii_itype}_features.csv"
        labelled_features = pd.read_csv(features_path, index_col=0)
        
        match extract_method:
            case "pyradiomics":
                features_only = getOnlyPyradiomicsFeatures(labelled_features)
            case _:
                raise ValueError(f"Unsupported extract_method: {extract_method}")

        correlations = features_only.corr(method=correlation_method)
        corr_matrix_writer.save(correlations,
                                Readii_iType = readii_itype,
                                CorrelationMethod=correlation_method)

    if plot:
        self_corr_hm, save_path = make_corr_figure(dataset,
                                                   correlations,
                                                   correlation_method=correlation_method,
                                                   cmap=plot_cmap,
                                                   x_subset_keyword=plot_subset_keyword,
                                                   y_subset_keyword=plot_subset_keyword,
                                                   include_volume=plot_include_volume,
                                                   correlation_type="self",
                                                   x_readii_itype=readii_itype,
                                                   signature=plot_signature,
                                                   overwrite=plot_overwrite)
        print(f"Correlation heatmap saved at {save_path}")


    return correlations, self_corr_hm if plot else None

In [8]:
def cross_correlate(dataset: str,
                   correlation_method:str,
                   x_readii_itype:str = "original_full",
                   y_readii_itype:str = "shuffled_full",
                   overwrite:bool = False,
                   plot:bool = False,
                   plot_cmap:str = 'coolwarm',
                   x_subset_keyword:str = None,
                   y_subset_keyword:str = None,
                   plot_include_volume:bool = False,
                   plot_signature:str | None = None,
                   plot_overwrite:bool = False
                   ) -> pd.DataFrame | tuple[pd.DataFrame, plt.Figure]:
    dataset_config, dataset_name, full_dataset_name = load_dataset_config(dataset)
    extract_method = dataset_config['EXTRACTION']['METHOD']
    extract_settings = dataset_config['EXTRACTION']['CONFIG']

    # Set up CorrelationWriter from readii
    corr_matrix_writer = CorrelationWriter(root_directory = dirs.RESULTS / full_dataset_name / "correlation" / "cross" / extract_method / extract_settings,
                                           filename_format = "{Y_iType}_vs_{X_iType}_{CorrelationMethod}.csv",
                                           overwrite = overwrite,
                                           create_dirs = True
    )

    corr_matrix_outpath = corr_matrix_writer.resolve_path(Y_iType=y_readii_itype,X_iType=x_readii_itype, CorrelationMethod=correlation_method)

    if corr_matrix_outpath.exists() and not overwrite:
        print(f"Correlation matrix already exists at {corr_matrix_outpath}. Skipping computation, loading existing matrix.")
        correlations = pd.read_csv(corr_matrix_outpath, index_col=0)
    
    else:
        print(f"Computing correlation matrix for {dataset_name} with method {correlation_method}...")

        features_path = dirs.RESULTS / full_dataset_name / "features" / extract_method / 'original_512_512_n' / extract_settings

        x_labelled_features = pd.read_csv(features_path / f"{x_readii_itype}_features.csv", index_col = 0)
        y_labelled_features = pd.read_csv(features_path / f"{y_readii_itype}_features.csv", index_col=0)

        match extract_method:
            case "pyradiomics":
                x_features_only = getOnlyPyradiomicsFeatures(x_labelled_features)
                y_features_only = getOnlyPyradiomicsFeatures(y_labelled_features)
            case _:
                raise ValueError(f"Unsupported extract_method: {extract_method}")

        # Combine x and y features into a single dataframe to compute correlations
        cross_features = x_features_only.join(y_features_only,
                                              how='inner',
                                              lsuffix='_x',
                                              rsuffix='_y')
        
        # This correlation matrix will have self and cross correlations contained in it in each quadrant
        mass_correlations = cross_features.corr(method=correlation_method)
        # Select out just the cross correlations between the specified image types in the correct axes
        correlations = mass_correlations.filter(like='_x', axis=1).filter(like='_y', axis=0)
        # Remove the _x and _y suffixes on the feature names
        correlations.columns = correlations.columns.str.replace("_x", "")
        correlations.index = correlations.index.str.replace("_y", "")
        # Save out the correlation matrix
        corr_matrix_writer.save(correlations,
                                Y_iType=y_readii_itype,
                                X_iType=x_readii_itype,
                                CorrelationMethod=correlation_method)

    if plot:
        cross_corr_hm, save_path = make_corr_figure(dataset,
                                                    correlations,
                                                    correlation_method=correlation_method,
                                                    cmap=plot_cmap,
                                                    x_subset_keyword=x_subset_keyword,
                                                    y_subset_keyword=y_subset_keyword,
                                                    include_volume=plot_include_volume,
                                                    correlation_type="cross",
                                                    x_readii_itype=x_readii_itype,
                                                    y_readii_itype=y_readii_itype,
                                                    signature=plot_signature,
                                                    overwrite=plot_overwrite)
        print(f"Correlation heatmap saved at {save_path}")

    return correlations, cross_corr_hm if plot else None

# Data Processing

In [ ]:
dataset = 'HEAD-NECK-RADIOMICS-HN1_windowed'

subset_keyword = 'original'

cross_corr_pd, cross_corr_hm = cross_correlate(dataset,
                                               correlation_method = "pearson",
                                               x_readii_itype = "original_full",
                                               y_readii_itype = "shuffled_full",
                                               plot_include_volume = False,
                                               plot = True,
                                               plot_cmap = 'plasma',
                                               x_subset_keyword=subset_keyword,
                                               y_subset_keyword=subset_keyword,
                                               plot_overwrite=False)

In [ ]:
dataset = 'RADCURE_windowed'
dataset_config, dataset_name, full_dataset_name = load_dataset_config(dataset)

subset_keyword = 'original'

for itype in image_type_iterator(dataset_config):
    readii_itype = "_".join(itype)
    self_corr_pd, self_corr_hm = self_correlate(dataset_name,
                                                correlation_method = "pearson",
                                                readii_itype = readii_itype,
                                                plot_include_volume = False,
                                                plot = True,
                                                plot_cmap = 'plasma',
                                                plot_subset_keyword=subset_keyword,
                                                plot_overwrite=False)
    plt.close()

    if readii_itype != "original_full":
        cross_corr_pd, cross_corr_hm = cross_correlate(dataset,
                                                correlation_method = "pearson",
                                                x_readii_itype = "original_full",
                                                y_readii_itype = readii_itype,
                                                plot_include_volume = False,
                                                plot = True,
                                                plot_cmap = 'plasma',
                                                x_subset_keyword=subset_keyword,
                                                y_subset_keyword=subset_keyword,
                                                plot_overwrite=False)

    plt.close()

In [ ]:
dataset = 'RADCURE_windowed'
dataset_config, dataset_name, full_dataset_name = load_dataset_config(dataset)

signature_name = 'choi_opc_os_2020'
signature_df = load_signature_config(signature_name)
signature_features = signature_df.index.append(pd.Index(['original_shape_MeshVolume']))

correlation_method = "pearson"
cmap = 'plasma'

for itype in image_type_iterator(dataset_config):
    readii_itype = "_".join(itype)
    self_corr_pd, _ = self_correlate(dataset_name,
                                  correlation_method = correlation_method,
                                  readii_itype = readii_itype,
                                  plot_include_volume = False,
                                  plot = False)
    
    self_signature_corrs = self_corr_pd.loc[signature_features,signature_features]

    self_corr_hm, save_path = make_corr_figure(dataset,
                                                self_signature_corrs,
                                                correlation_method=correlation_method,
                                                cmap=cmap,
                                                correlation_type="self",
                                                x_readii_itype=readii_itype,
                                                signature=signature_name,
                                                overwrite=False)
    plt.close()


    if readii_itype != "original_full":
        cross_corr_pd, _  = cross_correlate(dataset,
                                        correlation_method = correlation_method,
                                        x_readii_itype = "original_full",
                                        y_readii_itype = readii_itype,
                                        plot_include_volume = False,
                                        plot = False)
        
        cross_signature_corrs= cross_corr_pd.loc[signature_features,signature_features]

        cross_corr_hm, save_path = make_corr_figure(dataset,
                                                    cross_signature_corrs,
                                                    correlation_method=correlation_method,
                                                    cmap=cmap,
                                                    correlation_type="cross",
                                                    x_readii_itype="original_full",
                                                    y_readii_itype=readii_itype,
                                                    signature=signature_name,
                                                    overwrite=False)
    
    plt.close()

    

# Plot average correlation across 3 datasets

In [9]:
dataset = 'RADCURE_windowed'
dataset_config, dataset_name, full_dataset_name = load_dataset_config(dataset)

2026-05-01 | WARNING | Environment variable 'CONFIG' is not set. Using default path relative to project root.


### Self-correlations for each image type

In [ ]:
dataset = 'avg_RADCURE_HN1_NSCLC'
corr_type = "self"
subset_keyword = 'original'
correlation_method = "pearson"
cmap = "viridis"
signature = "Wavelet-HLH"


for itype in image_type_iterator(dataset_config):
    image_type = "_".join(itype)
    
    corr_file = Path("correlation") / corr_type / "pyradiomics" / "linear_all_images_features" / f"{image_type}_pearson.csv"

    radcure_correlations = pd.read_csv(dirs.RESULTS / "TCIA_RADCURE_windowed" / corr_file, index_col=0)
    hn1_correlations = pd.read_csv(dirs.RESULTS / "TCIA_HEAD-NECK-RADIOMICS-HN1_windowed" / corr_file, index_col=0)
    nsclc_correlations = pd.read_csv(dirs.RESULTS / "TCIA_NSCLC-Radiomics_windowed" / corr_file, index_col=0)

    average_correlations = np.abs((radcure_correlations + hn1_correlations + nsclc_correlations) / 3)

    avg_corr_subset = get_pattern_subset_features(average_correlations,
                                                  x_subset_keyword = subset_keyword,
                                                  y_subset_keyword = subset_keyword,
                                                  include_volume = False,
                                                  include_shape=True)

    self_corr_hm = plotCorrelationHeatmap(avg_corr_subset,
                                          diagonal = True,
                                          triangle = "lower",
                                          cmap = cmap,
                                          xlabel = f'{image_type} {signature} Features',
                                          ylabel = f'{image_type} {signature} Features',
                                          title = f"{corr_type.capitalize()}-Correlation for {dataset}",
                                          subtitle = f"{' '.join(image_type.split('_'))}",
                                          show_tick_labels = False,
                                          vmin = 0,
                                          vmax = 1.0
                                          )
    self_corr_hm.set_size_inches(12,10)

    self_corr_hm = add_pyrad_feature_class_labels(self_corr_hm)

    # save_path = saveCorrelationHeatmap(self_corr_hm, 
    #                                    correlation_directory = dirs.RESULTS / dataset / "visualization" / "feature_correlation" / corr_type / signature if signature else "",
    #                                    cmap = cmap,
    #                                    feature_types=[f'{image_type}'],
    #                                    correlation_type=correlation_method,
    #                                    overwrite = True)
    
    # plt.close()
    

### Plot each negative control correlations against the original image features

In [ ]:
dataset = 'avg_RADCURE_HN1_NSCLC'
corr_type = "cross"
correlation_method = "pearson"
cmap = "viridis"
# subset_keyword = 'wavelet-hlh'
# signature = "Wavelet-HLH"

# for subset_keyword in ['square_', 'squareroot', 'logarithm', 'exponential', 'gradient']:
#     signature = subset_keyword.capitalize()

# for subset_keyword in ['wavelet-LHL', 'wavelet-LHH', 'wavelet-HLL', 'wavelet-LLH', 'wavelet-HLH', 'wavelet-HHH', 'wavelet-HHL', 'wavelet-LLL']:
for subset_keyword in ['original']:
    signature = subset_keyword

    for itype in image_type_iterator(dataset_config):
        image_type = "_".join(itype)
        if image_type == 'original_full':
            # Load the original full self-correlation to look at
            corr_file = Path("correlation") / "self" / "pyradiomics" / "linear_all_images_features" / f"{image_type}_pearson.csv"

        else:
            corr_file = Path("correlation") / corr_type / "pyradiomics" / "linear_all_images_features" / f"{image_type}_vs_original_full_pearson.csv"

        radcure_correlations = pd.read_csv(dirs.RESULTS / "TCIA_RADCURE_windowed" / corr_file, index_col=0)
        hn1_correlations = pd.read_csv(dirs.RESULTS / "TCIA_HEAD-NECK-RADIOMICS-HN1_windowed" / corr_file, index_col=0)
        nsclc_correlations = pd.read_csv(dirs.RESULTS / "TCIA_NSCLC-Radiomics_windowed" / corr_file, index_col=0)
        
        average_correlations = np.abs((radcure_correlations + hn1_correlations + nsclc_correlations) / 3)

        avg_corr_subset = get_pattern_subset_features(average_correlations,
                                                    x_subset_keyword = subset_keyword,
                                                    y_subset_keyword = subset_keyword,
                                                    include_volume = False,
                                                    include_shape=True)                                

        cross_corr_hm = plotCorrelationHeatmap(avg_corr_subset,
                            diagonal = False,
                            cmap = cmap,
                            xlabel = f'original_full {signature} Features',
                            ylabel = f'{image_type} {signature} Features',
                            title = f"{corr_type.capitalize()}-Correlation for {dataset}",
                            subtitle = f"{' '.join(image_type.split('_'))} vs original full",
                            show_tick_labels = False,
                            vmin = 0,
                            vmax = 1.0
                            )
        cross_corr_hm.set_size_inches(12,10)

        cross_corr_hm = add_pyrad_feature_class_labels(cross_corr_hm)

        # save_path = saveCorrelationHeatmap(cross_corr_hm, 
        #                                 correlation_directory = dirs.RESULTS / dataset / "visualization" / "feature_correlation" / corr_type / signature if signature else "",
        #                                 cmap = cmap,
        #                                 feature_types=[f'{image_type}_vs_original_full'],
        #                                 correlation_type=correlation_method,
        #                                 overwrite = False)
        
        # plt.close()

### Plot difference between original vs. negative control correlations and original vs. original correlations

In [ ]:
dataset = 'avg_RADCURE_HN1_NSCLC'
corr_type = "cross"
subset_keyword = 'original'
correlation_method = "pearson"
cmap = "viridis"
signature = "Original"
absolute = False

for itype in image_type_iterator(dataset_config):
    image_type = "_".join(itype)
    if image_type == 'original_full':
        # Load the original full self-correlation to look at
        corr_file = Path("correlation") / "self" / "pyradiomics" / "linear_all_images_features" / f"{image_type}_pearson.csv"

    else:
        corr_file = Path("correlation") / corr_type / "pyradiomics" / "linear_all_images_features" / f"{image_type}_vs_original_full_pearson.csv"


    radcure_correlations = pd.read_csv(dirs.RESULTS / "TCIA_RADCURE_windowed" / corr_file, index_col=0)
    hn1_correlations = pd.read_csv(dirs.RESULTS / "TCIA_HEAD-NECK-RADIOMICS-HN1_windowed" / corr_file, index_col=0)
    nsclc_correlations = pd.read_csv(dirs.RESULTS / "TCIA_NSCLC-Radiomics_windowed" / corr_file, index_col=0)
    
    average_correlations = (radcure_correlations + hn1_correlations + nsclc_correlations) / 3

    orig_corr_file = Path("correlation") / "self" / "pyradiomics" / "linear_all_images_features" / f"original_full_pearson.csv"

    original_radcure_correlations = pd.read_csv(dirs.RESULTS / "TCIA_RADCURE_windowed" / orig_corr_file, index_col=0)
    original_hn1_correlations = pd.read_csv(dirs.RESULTS / "TCIA_HEAD-NECK-RADIOMICS-HN1_windowed" / orig_corr_file, index_col=0)
    original_nsclc_correlations = pd.read_csv(dirs.RESULTS / "TCIA_NSCLC-Radiomics_windowed" / orig_corr_file, index_col=0)

    original_average_correlations = (original_radcure_correlations + original_hn1_correlations + original_nsclc_correlations) / 3

    difference_correlations = average_correlations - original_average_correlations

    if absolute:
        difference_correlations = np.abs(difference_correlations)
        vmin = 0
    else:
        vmin = -1.0

    diff_corr_subset = get_pattern_subset_features(difference_correlations,
                                                   x_subset_keyword = subset_keyword,
                                                   y_subset_keyword = subset_keyword,
                                                   include_volume = False)
    
    diff_corr_hm = plotCorrelationHeatmap(diff_corr_subset,
                            diagonal = False,
                            cmap = cmap,
                            xlabel = f'original_full {signature} Features',
                            ylabel = f'{image_type} {signature} Features',
                            title = f"Difference in {corr_type.capitalize()}-Correlation for {dataset}",
                            subtitle = f"{' '.join(image_type.split('_'))} vs original full",
                            show_tick_labels = False,
                            vmin = vmin,
                            vmax = 1.0
                            )
    diff_corr_hm.set_size_inches(12,10)
    diff_corr_hm = add_pyrad_feature_class_labels(diff_corr_hm)
    
    # save_path = saveCorrelationHeatmap(diff_corr_hm,
    #                                     correlation_directory = dirs.RESULTS / dataset / "visualization" / "feature_correlation" / f"{corr_type}_corr_difference" / ("absolute" if absolute else "relative") / signature if signature else "",
    #                                     cmap = cmap,
    #                                     feature_types=[f'{image_type}_vs_original_full'],
    #                                     correlation_type=correlation_method,
    #                                     overwrite = True)
    
    # plt.close()

### Plot average correlations of 3 datasets for specific signatures + MeshVolume

In [ ]:
dataset = 'avg_RADCURE_HN1'
corr_type = "cross"
subset_keyword = 'original'
correlation_method = "pearson"
cmap = "viridis"
absolute = False
signature = 'choi_opc_os_2020'

signature_df = load_signature_config(signature)
signature_features = signature_df.index.append(pd.Index(['original_shape_MeshVolume']))

for itype in image_type_iterator(dataset_config):
    image_type = "_".join(itype)
    if image_type == 'original_full':
        # Load the original full self-correlation to look at
        corr_file = Path("correlation") / "self" / "pyradiomics" / "linear_all_images_features" / f"{image_type}_pearson.csv"

    else:
        corr_file = Path("correlation") / corr_type / "pyradiomics" / "linear_all_images_features" / f"{image_type}_vs_original_full_pearson.csv"

    radcure_correlations = pd.read_csv(dirs.RESULTS / "TCIA_RADCURE_windowed" / corr_file, index_col=0)
    hn1_correlations = pd.read_csv(dirs.RESULTS / "TCIA_HEAD-NECK-RADIOMICS-HN1_windowed" / corr_file, index_col=0)
    # nsclc_correlations = pd.read_csv(dirs.RESULTS / "TCIA_NSCLC-Radiomics_windowed" / corr_file, index_col=0)
    
    average_correlations = (radcure_correlations + hn1_correlations) / 2 # + nsclc_correlations) / 3

    signature_correlations = average_correlations.loc[signature_features,signature_features]

    if absolute:
        signature_correlations = np.abs(signature_correlations)
        vmin = 0
    else:
        vmin = -1.0

    signature_corr_hm = plotCorrelationHeatmap(signature_correlations,
                            diagonal = False,
                            cmap = cmap,
                            xlabel = f'original_full {signature} Features',
                            ylabel = f'{image_type} {signature} Features',
                            title = f"{signature} Signature Correlations for {dataset}",
                            subtitle = f"{' '.join(image_type.split('_'))} vs original full",
                            show_tick_labels = True,
                            vmin = vmin,
                            vmax = 1.0
                            )
    signature_corr_hm.set_size_inches(12,10)
    signature_corr_hm.axes[0].tick_params(axis='both', labelsize=9)

    save_path = saveCorrelationHeatmap(signature_corr_hm,
                                        correlation_directory = dirs.RESULTS / dataset / "visualization" / "feature_correlation" / corr_type / (signature if signature else "")/ ("absolute" if absolute else "relative"),
                                        cmap = cmap,
                                        feature_types=[f'{image_type}_vs_original_full'],
                                        correlation_type=correlation_method,
                                        overwrite = True)
    
    plt.close()
